# 第7章：深入 Keras —— 三种模型构建方法与训练扩展

**学习路线：**

| 阶段 | 章节 | 内容 | 关键词 |
|------|------|------|--------|
| **一、模型构建** | §1-§3 | Sequential、函数式API、模型子类化 | Sequential, Functional API, Model Subclassing |
| **进阶示例** | §4 | 三种方法对比与选择指南 | 对比表 |
| **二、内置训练** | §5-§6 | MNIST全流程、自定义指标、回调函数 | compile, fit, callbacks, custom metric |

> **学习建议**：建议从 §1 开始按顺序运行。§2.5-§2.10（多输入多输出、特征复用）是函数式 API 的核心优势，务必掌握。
>
> 本笔记本使用 MNIST 真实数据集和合成数据混合演示，所有代码可直接运行。

In [ ]:
# === 基础导入 ===
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")

---
# 第一阶段：模型构建

本阶段学习 Keras 的三种建模方法，
掌握从简单到复杂的模型搭建方式：Sequential、函数式 API、模型子类化。

---
## 7.1 Keras 工作流程

Keras 的核心设计原则：**渐进式呈现复杂性**（Progressive Disclosure of Complexity）。

这意味着：
- 对初学者：可以从最简单的 API 开始，快速搭建能工作的模型
- 对进阶用户：当简单 API 不够用时，可以无缝切换到更灵活的方式

三种模型构建方法，从简单到灵活：

| 方法 | 复杂度 | 适用场景 |
|------|--------|----------|
| **序贯模型（Sequential）** | 最易理解 | 层的简单堆叠，一条线结构 |
| **函数式API（Functional API）** | 平衡灵活 | 多输入/多输出/残差连接，最常用 |
| **模型子类化（Model Subclassing）** | 最底层 | 完全控制前向传播逻辑 |

---
## §1 序贯模型（Sequential Model）

Sequential 是 Keras 最简单的建模方式 —— 本质上是**一个 Python 列表**，
所有层按顺序排列，数据从第一层流到最后一层。

**结构特征**：只能处理"一条线"的结构
```
Input → Layer1 → Layer2 → Layer3 → Output
```

**局限性**：不支持多输入、多输出、残差连接、共享层等复杂拓扑。

### 1.1 方式一：列表式初始化

最直观的写法 —— 在构造时传入层的列表，一行代码搞定。

In [ ]:
# 列表式初始化：把所有层放在一个列表中传入
model = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

# ⚠️ 注意：此时模型还没有权重！
# Keras 使用"延迟构建"策略 —— 只有收到实际输入时才创建权重
print("模型创建成功，但此时还没有权重（延迟构建）")

### 1.2 方式二：逐步 add() 添加层

先创建空模型，再一层一层添加。适合在运行时动态决定层结构的场景。

In [ ]:
# 逐步添加：先创建空模型，再逐层 add()
model = keras.Sequential()
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(10, activation="softmax"))

print("通过 add() 逐层构建完成")
print(f"模型包含 {len(model.layers)} 层")

### 1.3 Sequential 模型的权重延迟创建机制

Keras 的 **延迟构建（Deferred Build）** 策略是理解 Keras 的关键概念之一：

| 时机 | 状态 |
|------|------|
| `Sequential([...])` 刚创建 | 权重**未创建**，访问会报 `ValueError` |
| 第一次调用 `model(x)` 或调用 `build()` | 权重**已创建** |

**好处**：不需要在构造时就指定输入维度，Keras 自动推断。

In [ ]:
# 演示延迟构建：尝试在 build 之前访问权重
model = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

try:
    print(model.weights)  # 会抛出 ValueError
except ValueError as e:
    print(f"[预期错误] 权重未创建:\n  {e}")

# 手动触发构建：调用 build() 并指定输入形状
# input_shape 中的 None 表示批量大小可以是任意值
model.build(input_shape=(None, 3))

# 现在权重已创建，可以访问了
print(f"\n[build 之后] 权重数量: {len(model.weights)}")
for w in model.weights:
    print(f"  {w.name}: shape={w.shape}")

### 1.4 使用 build() 后查看模型结构和权重

`model.summary()` 打印模型摘要：
- 每层的输出形状
- 每层的参数数量
- 总参数量（可训练 + 不可训练）

In [ ]:
# build 后查看模型摘要
model.build(input_shape=(None, 3))
model.summary()

# 参数计算验证：
# Dense(64): 3 * 64 + 64 = 256 (权重 + 偏置)
# Dense(10): 64 * 10 + 10 = 650
# 总计: 256 + 650 = 906

### 1.5 为层和模型命名

通过 `name` 参数指定名称，在 `summary()` 和调试时更容易识别。

In [ ]:
# 为模型和每一层指定名称
model = keras.Sequential([
    layers.Dense(64, activation="relu", name="feature_extractor"),
    layers.Dense(10, activation="softmax", name="classifier")
], name="my_sequential_model")

model.build(input_shape=(None, 3))
model.summary()

# 观察 summary 中 "Output Shape" 列：
# (None, 64) 和 (None, 10) —— None 表示批量维度

### 1.6 Sequential 模型的动态增删特性

可以在构建前后随时 `add()` 新层，`summary()` 会实时反映当前结构。

In [ ]:
# 动态增删层：在已有模型上添加新层
model = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

print("=== 添加新层之前 ===")
model.build(input_shape=(None, 3))
model.summary()

# 在末尾添加一个新层
model.add(layers.Dense(5, activation="softmax"))

print("\n=== 添加新层之后 ===")
model.summary()

# 注意：添加新层后参数量从 906 增加到 1061
# 新增参数: 10 * 5 + 5 = 55

### 1.7 使用 Input 对象声明输入形状（替代 build()）

更推荐的方式：用 `keras.Input()` 替代 `build()`。

| 方式 | 语法 | 风格 |
|------|------|------|
| `build()` | `model.build(input_shape=(None, 3))` | 命令式 |
| `Input()` | `model.add(keras.Input(shape=(3,)))` | 声明式，**推荐** |

In [ ]:
# 使用 Input 对象声明输入形状（推荐方式）
model = keras.Sequential()
model.add(keras.Input(shape=(3,)))  # shape 是单个样本的形状，不包含批量维度
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(10, activation="softmax"))

model.summary()

# 对比：
# - Input(shape=(3,)) → 输入形状 (None, 3)，None 表示批量维度
# - build(input_shape=(None, 3)) → 效果相同，但 Input 更声明式、更清晰

### 第一阶段小结：Sequential

| 概念 | 要点 |
|------|------|
| 列表式初始化 | `Sequential([层1, 层2, ...])` 一行搞定 |
| 逐步 add() | 适合运行时动态决定层结构 |
| 延迟构建 | 权重在第一次收到输入时才创建 |
| `build()` | 手动触发权重创建 |
| `Input()` | 声明式指定输入形状，**推荐** |
| 命名 | 通过 `name` 参数方便调试 |

---
## §2 函数式 API（Functional API）

函数式 API 是 **最常用的构建方式**，在可用性和灵活性之间找到平衡。

**核心思想**：**层被当作函数来调用**，输入是张量，输出也是张量。

与 Sequential 的区别：
```
Sequential:    Input → L1 → L2 → L3 → Output  (一条线)

函数式 API:     ┌───── Input ─────┐
                │                 │
              Branch A         Branch B
                │                 │
                └──→ Merge ←─────┘
                       │
                    Output
```

可以处理：残差连接、多输入多输出、共享层、非DAG结构。

### 2.1 函数式 API 核心用法：单输入单输出

**三板斧**：定义输入 → 定义层连接 → 打包模型

In [ ]:
# 函数式 API 三板斧

# 1. 定义输入节点（张量的起点）
inputs = keras.Input(shape=(3,), name="my_input")

# 2. 把层当函数调用，传入上一层的输出
# 语法：Layer(...)(上一层的输出)
features = layers.Dense(64, activation="relu")(inputs)
outputs = layers.Dense(10, activation="softmax")(features)

# 3. 创建模型：指定 inputs 和 outputs，Keras 自动推导中间的计算图
model = keras.Model(inputs=inputs, outputs=outputs)

model.summary()

### 2.2 Input 对象的属性

`keras.Input()` 返回的是一个**符号张量**（Symbolic Tensor），不是真实数据。

In [ ]:
# 查看 Input 对象的属性
inputs = keras.Input(shape=(3,))
print(f"inputs.shape: {inputs.shape}")  # (None, 3) —— None 表示批量维度可变
print(f"inputs.dtype: {inputs.dtype}")  # 默认 float32

# 指定不同的 dtype
int_inputs = keras.Input(shape=(3,), dtype="int32")
print(f"int_inputs.dtype: {int_inputs.dtype}")

### 2.3 函数式 API 的数据流：层作为函数调用

每一层被当作函数来调用，返回一个新的输出张量。
可以通过查看中间张量形状来理解数据在模型中的流动。

In [ ]:
# 查看中间张量的形状 —— 理解数据流
inputs = keras.Input(shape=(784,))  # 例如：MNIST 展平后的图像
features = layers.Dense(64, activation="relu")(inputs)
outputs = layers.Dense(10, activation="softmax")(features)

print(f"输入张量形状:   {inputs.shape}    ← (None, 784)  784个像素"
      f"\n中间特征形状:   {features.shape}    ← (None, 64)   64维特征表示"
      f"\n输出张量形状:   {outputs.shape}    ← (None, 10)   10个类别的概率")

### 2.4 函数式 API 的模型概览（summary）

与 Sequential 不同，summary 中会显示 `InputLayer`，层的连接关系清晰可见。

In [ ]:
# 函数式 API 的 summary 会显示 InputLayer
inputs = keras.Input(shape=(784,), name="image_input")
x = layers.Dense(512, activation="relu", name="hidden1")(inputs)
x = layers.Dropout(0.5, name="dropout")(x)
outputs = layers.Dense(10, activation="softmax", name="output")(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="functional_demo")
model.summary()

# 注意：第一行显示 InputLayer，这是 Sequential 中不会出现的
# 它代表模型的输入节点

### 2.5 多输入多输出模型（函数式 API 的核心优势）

Sequential 做不到多输入多输出，但函数式 API 可以。

**场景**：工单分类器（ticket classifier）
- 三个输入：标题（title）、正文（text_body）、标签（tags）
- 两个输出：优先级（priority）、部门（department）

```
    title ──┐
text_body ──┼──→ Concatenate ──→ Dense(64) ──┬──→ Dense(1, sigmoid) → priority
    tags ───┘                                └──→ Dense(4, softmax) → department
```

In [ ]:
# 2.5.1 定义三个输入
vocabulary_size = 10000  # 词表大小
num_tags = 100           # 标签数量
num_departments = 4      # 部门数量

title = keras.Input(shape=(vocabulary_size,), name="title")
text_body = keras.Input(shape=(vocabulary_size,), name="text_body")
tags = keras.Input(shape=(num_tags,), name="tags")

In [ ]:
# 2.5.2 拼接三个输入特征，然后通过一个共享的全连接层
features = layers.Concatenate()([title, text_body, tags])  # 拼接: (None, 20100)
features = layers.Dense(64, activation="relu")(features)   # 压缩: (None, 64)

In [ ]:
# 2.5.3 从共享特征分出两个输出分支
priority = layers.Dense(1, activation="sigmoid", name="priority")(features)
department = layers.Dense(num_departments, activation="softmax", name="department")(features)

In [ ]:
# 2.5.4 创建多输入多输出模型
model = keras.Model(
    inputs=[title, text_body, tags],
    outputs=[priority, department]
)
model.summary()

### 2.6 多输入多输出模型的训练：列表式传参

输入、目标、loss、metrics 都以 **列表** 形式传入，顺序与模型定义一致。

In [ ]:
# 用合成数据演示多输入多输出模型的训练（列表式传参）
num_samples = 1280

# 构造输入数据（模拟 one-hot 编码的文本和标签）
title_data = np.random.randint(0, 2, (num_samples, vocabulary_size)).astype('float32')
text_body_data = np.random.randint(0, 2, (num_samples, vocabulary_size)).astype('float32')
tags_data = np.random.randint(0, 2, (num_samples, num_tags)).astype('float32')

# 构造目标数据
priority_data = np.random.random((num_samples, 1))         # 优先级：0~1 的连续值
department_data = np.random.randint(0, num_departments, num_samples)  # 部门：整数类别

# 编译模型：loss 和 metrics 以列表形式指定，顺序对应 outputs 列表
model.compile(
    optimizer="rmsprop",
    loss=["mean_squared_error", "sparse_categorical_crossentropy"],
    metrics=[["mean_absolute_error"], ["accuracy"]]
)

# 训练：输入和目标都是列表
model.fit(
    x=[title_data, text_body_data, tags_data],
    y=[priority_data, department_data],
    epochs=2,
    batch_size=32
)

### 2.7 多输入多输出模型的训练：字典式传参（推荐）

使用字典方式代码更清晰，不易出错。
键与 Input/输出层的 `name` 一一对应，**不需要记住顺序**。

In [ ]:
# 重新定义模型（确保 name 对齐）
title = keras.Input(shape=(vocabulary_size,), name="title")
text_body = keras.Input(shape=(vocabulary_size,), name="text_body")
tags = keras.Input(shape=(num_tags,), name="tags")

features = layers.Concatenate()([title, text_body, tags])
features = layers.Dense(64, activation="relu")(features)
priority = layers.Dense(1, activation="sigmoid", name="priority")(features)
department = layers.Dense(num_departments, activation="softmax", name="department")(features)

model = keras.Model(
    inputs=[title, text_body, tags],
    outputs=[priority, department]
)

# 编译模型：使用字典方式指定 loss 和 metrics
# 键名必须与输出层的 name 参数一致
model.compile(
    optimizer="rmsprop",
    loss={
        "priority": "mean_squared_error",
        "department": "sparse_categorical_crossentropy"
    },
    metrics={
        "priority": ["mean_absolute_error"],
        "department": ["accuracy"]
    }
)

# 训练：使用字典传参
model.fit(
    x={"title": title_data, "text_body": text_body_data, "tags": tags_data},
    y={"priority": priority_data, "department": department_data},
    epochs=2,
    batch_size=32
)

# 推荐字典传参的理由：
# 1. 不依赖顺序，键名自解释
# 2. 修改模型输出顺序时，不用改 fit() 的参数
# 3. 代码可读性更强

### 2.8 可视化模型结构：plot_model()

将模型结构保存为图片，方便理解复杂拓扑。

In [ ]:
# 可视化模型结构
try:
    keras.utils.plot_model(model, "ticket_classifier.png", show_shapes=True)
    print("模型图已保存为 ticket_classifier.png")
    from IPython.display import Image, display
    display(Image("ticket_classifier.png"))
except Exception as e:
    print(f"可视化需要安装 pydot 和 graphviz: {e}")
    print("可以跳过此步，不影响模型训练")

### 2.9 访问模型的层（layers 属性）

`model.layers` 返回模型中所有层的列表，可以像普通 Python 列表一样索引。

In [ ]:
# 访问模型的层
print(f"模型共有 {len(model.layers)} 层")
for i, layer in enumerate(model.layers):
    print(f"  [{i}] {layer.name}")

# 查看特定层的输入/输出张量
# layers[3] 是 Concatenate 层
concat_layer = model.layers[3]
print(f"\nConcatenate 层的输入: {concat_layer.input}")
print(f"Concatenate 层的输出: {concat_layer.output}")

### 2.10 从已有模型中提取特征创建新模型（特征复用）

函数式 API 最强大的特性之一：**从已有的模型中间层提取输出，构建新模型**。

场景：在已训练的工单分类器基础上，增加一个"难度"分类任务。

```text
原模型: inputs → Concatenate → Dense(64) → [priority, department]
新模型: inputs → Concatenate → Dense(64) → [priority, department, difficulty]
```

**关键**：新模型复用了原模型的权重（如果原模型已训练过）。

In [ ]:
# 从中间层提取输出，创建扩展模型

# 1. 从第 4 层（Dense 64，即共享特征层）提取输出张量
features = model.layers[4].output

# 2. 在已有特征上添加新输出层
difficulty = layers.Dense(3, activation="softmax", name="difficulty")(features)

# 3. 用相同的 inputs 和新的 outputs 创建新模型
extended_model = keras.Model(
    inputs=model.inputs,
    outputs=[model.outputs[0], model.outputs[1], difficulty]
)

extended_model.summary()

# 注意：extended_model 复用了 model 中 Concatenate 和 Dense(64) 的权重
# 如果原模型已经训练过，新模型的共享部分已经是有意义的特征表示

---
## §3 模型子类化（Model Subclassing）

最底层、最灵活的方式 —— **从头编写所有内容**。

与函数式 API 的根本区别：
- 函数式 API：声明式，Keras 知道整个计算图的结构
- 子类化：命令式，前向传播逻辑写在 `call()` 中，可以加入任意 Python 控制流（if/for/while）

**适用场景**：研究性代码、GAN、动态网络结构、需要复杂控制流的情况。

### 3.1 ~ 3.2 完整子类化模型示例

两步：
1. `__init__()` 中定义所有子层（`super().__init__()` 必不可少）
2. `call()` 中定义前向传播逻辑

In [ ]:
class CustomerTicketModel(keras.Model):
    """
    工单分类器的子类化实现。
    
    与函数式 API 版本的功能完全相同，但写法不同：
    - 所有层定义在 __init__ 中（相当于"声明零件"）
    - 前向传播逻辑写在 call() 中（相当于"组装流程"）
    - 可以加入任意 Python 控制流（if/for/while）
    """
    
    def __init__(self, num_departments):
        super().__init__()  # ⚠️ 必不可少！
        
        # 在 __init__ 中定义所有子层（声明零件）
        self.concat_layer = layers.Concatenate()
        self.mixing_layer = layers.Dense(64, activation="relu")
        self.priority_scorer = layers.Dense(1, activation="sigmoid")
        self.department_classifier = layers.Dense(num_departments, activation="softmax")

    def call(self, inputs):
        # 在 call() 中定义前向传播逻辑（组装流程）
        title = inputs["title"]
        text_body = inputs["text_body"]
        tags = inputs["tags"]
        
        features = self.concat_layer([title, text_body, tags])
        features = self.mixing_layer(features)
        priority = self.priority_scorer(features)
        department = self.department_classifier(features)
        
        return priority, department

### 3.3 子类化模型的实例化和调用

- 实例化时 **无需指定输入形状**
- 权重在 **第一次调用时** 才创建（动态构建）
- 直接像函数一样调用模型

In [ ]:
# 实例化模型（不需要指定输入形状）
model = CustomerTicketModel(num_departments=4)

# 此时还没有权重
print(f"[调用前] 权重数量: {len(model.weights)}")

# 构造虚拟输入数据
dummy_inputs = {
    "title": np.random.random((32, vocabulary_size)).astype('float32'),
    "text_body": np.random.random((32, vocabulary_size)).astype('float32'),
    "tags": np.random.random((32, num_tags)).astype('float32'),
}

# 第一次调用时才创建权重（动态构建）
priority, department = model(dummy_inputs)
print(f"\n[调用后] 权重数量: {len(model.weights)}")
print(f"priority 输出形状:   {priority.shape}    ← (32, 1)")
print(f"department 输出形状: {department.shape}  ← (32, 4)")

### 3.4 子类化模型的 compile、fit、evaluate、predict

子类化模型同样支持 Keras 的标准训练流程，但要注意：
- `loss` 和 `metrics` 的结构必须与 `call()` 返回值的结构完全匹配
- 输入数据的结构必须与 `call()` 方法的输入格式匹配

In [ ]:
# 编译和训练子类化模型
model = CustomerTicketModel(num_departments=4)

# ⚠️ 注意：loss 和 metrics 的顺序必须与 call() 返回值顺序一致
# call() 返回 (priority, department)，所以 loss 列表也要对应
model.compile(
    optimizer="rmsprop",
    loss=["mean_squared_error", "sparse_categorical_crossentropy"],
    metrics=[["mean_absolute_error"], ["accuracy"]]
)

# 训练数据
num_samples = 1280
train_inputs = {
    "title": np.random.randint(0, 2, (num_samples, vocabulary_size)).astype('float32'),
    "text_body": np.random.randint(0, 2, (num_samples, vocabulary_size)).astype('float32'),
    "tags": np.random.randint(0, 2, (num_samples, num_tags)).astype('float32'),
}
train_targets = [
    np.random.random((num_samples, 1)),          # priority: 连续值
    np.random.randint(0, 4, num_samples),         # department: 整数类别
]

# 训练：与函数式 API 完全相同的 fit() 接口
model.fit(train_inputs, train_targets, epochs=2, batch_size=32)

### 3.5 子类化模型与函数式 API 的结合使用

将子类化的 Model 当作函数式 API 中的一个"层"来使用。

In [ ]:
# 将子类化模型嵌入函数式 API 中

# 1. 创建一个子类化分类器实例
classifier = CustomerTicketModel(num_departments=4)

# 2. 在函数式 API 中把它当作一个"层"使用
title = keras.Input(shape=(vocabulary_size,), name="title")
text_body = keras.Input(shape=(vocabulary_size,), name="text_body")
tags = keras.Input(shape=(num_tags,), name="tags")

# 直接调用子类化模型（它会返回两个输出）
priority, department = classifier({
    "title": title,
    "text_body": text_body,
    "tags": tags
})

# 3. 创建函数式模型（把计算图打包）
hybrid_model = keras.Model(
    inputs=[title, text_body, tags],
    outputs=[priority, department]
)
hybrid_model.summary()

# 本质：Keras Model 也是 Layer 的子类，所以可以嵌套使用

### 3.6 在子类化模型中复用已有的函数式 API 模型

先用函数式 API 创建一个独立模型，然后在子类化模型的 `__init__` 中引用它。

In [ ]:
# 3.6.1 先用函数式 API 创建一个独立的二分类器
binary_classifier = keras.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="binary_scorer")

# 3.6.2 在子类化模型中复用该模型
class ComplexModel(keras.Model):
    """在子类化模型中嵌入已有的函数式 API 模型"""
    
    def __init__(self):
        super().__init__()
        # 将已有的模型作为组件引用
        self.binary_scorer = binary_classifier
        self.dense = layers.Dense(32, activation="relu")
    
    def call(self, inputs):
        x = self.dense(inputs)
        return self.binary_scorer(x)  # 调用嵌入的模型

# 测试
complex_model = ComplexModel()
output = complex_model(tf.random.normal((2, 100)))
print(f"输出形状: {output.shape}")  # (2, 1)

### 第一阶段小结

| 概念 | 要点 |
|------|------|
| Sequential | 简单模型，层按顺序堆叠，适合快速原型 |
| 函数式 API | 复杂模型，支持残差连接、多输入输出，**推荐首选** |
| 模型子类化 | 完全控制 call()，可加入任意 Python 控制流 |
| 特征复用 | 从已有模型中间层提取输出构建新模型 |
| 混合使用 | 子类化模型可以嵌入函数式 API，反之亦然 |

---
# 第二阶段：内置训练循环和评估循环

本阶段学习 Keras 的训练四步曲：**compile → fit → evaluate → predict**。

```text
┌─────────────────────────────────────────────────────────────┐
│  Step 1: compile(optimizer, loss, metrics)  — 配置训练规则  │
│  Step 2: fit(x, y, epochs, batch_size)      — 训练模型      │
│  Step 3: evaluate(x, y)                      — 评估性能      │
│  Step 4: predict(x)                          — 预测新数据    │
└─────────────────────────────────────────────────────────────┘
```

---
## §5 标准 Keras 工作流全流程（MNIST）

使用真实数据集 MNIST 演示完整流程。

In [ ]:
# 5.1 定义模型构建函数（便于复用）
def get_mnist_model():
    """
    构建 MNIST 手写数字分类模型。
    
    结构：Input → Dense(512, relu) → Dropout(0.5) → Dense(10, softmax)
    """
    inputs = keras.Input(shape=(28 * 28,))
    features = layers.Dense(512, activation="relu")(inputs)
    features = layers.Dropout(0.5)(features)  # 防止过拟合
    outputs = layers.Dense(10, activation="softmax")(features)
    model = keras.Model(inputs, outputs)
    return model

In [ ]:
# 5.2 数据加载与预处理
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

# reshape 为 (样本数, 28*28) 并归一化到 [0, 1]
# MNIST 原始像素值范围 0~255，除以 255 后归一化到 [0, 1]
train_images = train_images.reshape((60000, 28 * 28)).astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255

# 划分训练集和验证集（前 10000 个样本作验证）
x_val = train_images[:10000]
y_val = train_labels[:10000]
x_train = train_images[10000:]
y_train = train_labels[10000:]

print(f"训练集: {x_train.shape[0]} 样本, 形状 {x_train.shape}")
print(f"验证集: {x_val.shape[0]} 样本, 形状 {x_val.shape}")
print(f"测试集:  {test_images.shape[0]} 样本, 形状 {test_images.shape}")

In [ ]:
# 5.3 编译模型
model = get_mnist_model()
model.compile(
    optimizer="rmsprop",                    # 优化器：RMSprop
    loss="sparse_categorical_crossentropy", # 损失函数：整数标签的多分类交叉熵
    metrics=["accuracy"]                    # 监控指标：准确率
)
model.summary()

In [ ]:
# 5.4 训练模型（fit）
history = model.fit(
    x_train, y_train,
    epochs=5,
    validation_data=(x_val, y_val)
)

# fit() 返回 History 对象，记录每轮的 loss、accuracy、val_loss、val_accuracy
# 可以通过 history.history 访问这些数据

In [ ]:
# 5.5 评估模型（evaluate）
# 在独立测试集上评估，不更新参数
test_metrics = model.evaluate(test_images, test_labels, verbose=0)
print(f"测试集 - loss: {test_metrics[0]:.4f}, accuracy: {test_metrics[1]:.4f}")

In [ ]:
# 5.6 使用模型预测（predict）
predictions = model.predict(test_images[:5])

print("前 5 张测试图片的预测结果：")
print(f"{'样本':>6} | {'预测类别':>6} | {'概率':>8} | {'真实标签':>6}")
print("-" * 40)
for i, pred in enumerate(predictions):
    pred_class = np.argmax(pred)
    pred_prob = pred[pred_class]
    true_label = test_labels[i]
    match = "✓" if pred_class == true_label else "✗"
    print(f"  #{i:>4} | {pred_class:>6} | {pred_prob:>8.4f} | {true_label:>6} {match}")

---
## §6 自定义指标（Custom Metrics）

当内置指标（accuracy、loss 等）不满足需求时，可以自定义。

**实现步骤**：继承 `keras.metrics.Metric` 类，实现四个方法：

| 方法 | 调用时机 | 职责 |
|------|---------|------|
| `__init__()` | 实例化时 | 定义状态变量（`self.add_weight()`） |
| `update_state()` | 每个 batch 结束时 | 更新状态的计算逻辑 |
| `result()` | 需要读取指标值时 | 返回当前的指标值 |
| `reset_state()` | 每个 epoch 开始时 | 重置状态变量 |

In [ ]:
class RootMeanSquaredError(keras.metrics.Metric):
    """
    自定义 RMSE（均方根误差）指标。
    
    数学公式：RMSE = sqrt( Σ(y_true - y_pred)² / N )
    
    计算逻辑：
    1. 每个 batch 中：累加 MSE 值和样本数量
    2. result()：返回 sqrt(总MSE / 总样本数)
    3. 每个 epoch 开始：重置累加器
    """
    
    def __init__(self, name="rmse", **kwargs):
        super().__init__(name=name, **kwargs)
        # 定义两个状态变量：MSE 累加器 + 样本数累加器
        self.mse_sum = self.add_weight(name="mse_sum", initializer="zeros")
        self.total_samples = self.add_weight(
            name="total_samples", initializer="zeros", dtype="int32")

    def update_state(self, y_true, y_pred, sample_weight=None):
        # 将整数标签转为 one-hot 编码，以便与预测概率计算 MSE
        # y_pred shape: (batch_size, 10), y_true shape: (batch_size,)
        y_true = tf.one_hot(y_true, depth=tf.shape(y_pred)[1])
        mse = tf.reduce_sum(tf.square(y_true - y_pred))
        self.mse_sum.assign_add(mse)
        num_samples = tf.shape(y_pred)[0]
        self.total_samples.assign_add(num_samples)

    def result(self):
        return tf.sqrt(self.mse_sum / tf.cast(self.total_samples, tf.float32))

    def reset_state(self):
        self.mse_sum.assign(0.)
        self.total_samples.assign(0)

In [ ]:
# 在 compile 中使用自定义指标
model = get_mnist_model()
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", RootMeanSquaredError()]  # 内置指标 + 自定义指标
)

# 训练：观察自定义 RMSE 指标在每个 epoch 的输出
# 你会看到进度条中除了 accuracy 还多了一个 rmse
model.fit(
    x_train, y_train,
    epochs=3,
    validation_data=(x_val, y_val)
)

---
## §7 回调函数（Callbacks）

回调函数在训练过程的特定时间点自动执行自定义操作。

**6 个核心钩子方法**：

| 钩子 | 调用时机 |
|------|---------|
| `on_epoch_begin(epoch, logs)` | 每轮训练开始时 |
| `on_epoch_end(epoch, logs)` | 每轮训练结束时 |
| `on_batch_begin(batch, logs)` | 每个 batch 处理前 |
| `on_batch_end(batch, logs)` | 每个 batch 处理后 |
| `on_train_begin(logs)` | 整个训练开始时 |
| `on_train_end(logs)` | 整个训练结束时 |

**常用内置回调**：

| Callback | 作用 |
|----------|------|
| `EarlyStopping` | 验证指标不再改善时提前停止 |
| `ModelCheckpoint` | 自动保存最佳模型权重 |
| `ReduceLROnPlateau` | 验证指标停滞时降低学习率 |
| `TensorBoard` | 记录训练日志供可视化 |

### 7.1 EarlyStopping + ModelCheckpoint 组合使用

In [ ]:
# 组合使用 EarlyStopping + ModelCheckpoint
callbacks_list = [
    # 早停：验证精度连续 2 轮不改善则中断训练
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=2,
    ),
    # 模型检查点：仅在验证损失改善时保存模型
    keras.callbacks.ModelCheckpoint(
        filepath="checkpoint_path.keras",
        monitor="val_loss",
        save_best_only=True,  # 只保存最优模型，覆盖旧文件
    )
]

model = get_mnist_model()
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 训练：最多 10 轮，但 EarlyStopping 可能提前终止
model.fit(
    x_train, y_train,
    epochs=10,
    validation_data=(x_val, y_val),
    callbacks=callbacks_list
)

In [ ]:
# 加载保存的最优模型
try:
    best_model = keras.models.load_model("checkpoint_path.keras")
    print("[成功] 最优模型已加载")
    
    # 在测试集上评估最优模型
    test_loss, test_acc = best_model.evaluate(test_images, test_labels, verbose=0)
    print(f"最优模型的测试精度: {test_acc:.4f}")
except Exception as e:
    print(f"[提示] 加载模型失败（可能 EarlyStopping 在前几轮就终止了）: {e}")

### 7.2 自定义回调函数：LossHistory

继承 `keras.callbacks.Callback`，实现感兴趣的钩子方法。

**场景**：记录每个 batch 的损失值，每轮结束后绘制损失曲线。

In [ ]:
from matplotlib import pyplot as plt

class LossHistory(keras.callbacks.Callback):
    """
    自定义回调：记录每个 batch 的损失，并在每个 epoch 结束时绘图。
    
    用途：观察每个 batch 的损失变化趋势，
    帮助判断学习率是否合适、训练是否稳定。
    """
    
    def on_train_begin(self, logs=None):
        self.per_batch_losses = []

    def on_batch_end(self, batch, logs=None):
        # logs 是一个字典，包含 'loss', 'accuracy' 等
        self.per_batch_losses.append(logs.get("loss"))

    def on_epoch_end(self, epoch, logs=None):
        plt.clf()
        plt.plot(range(len(self.per_batch_losses)), self.per_batch_losses,
                 label="Training loss for each batch")
        plt.xlabel(f"Batch (epoch {epoch})")
        plt.ylabel("Loss")
        plt.legend()
        plt.savefig(f"plot_at_epoch_{epoch}.png")
        plt.close()
        self.per_batch_losses = []  # 清空，为下一个 epoch 重新开始记录
        print(f"  [Epoch {epoch}] 损失曲线已保存: plot_at_epoch_{epoch}.png")

In [ ]:
# 使用自定义回调
model = get_mnist_model()
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

loss_history = LossHistory()
model.fit(
    x_train, y_train,
    epochs=3,
    validation_data=(x_val, y_val),
    callbacks=[loss_history]
)

### 7.3 TensorBoard 回调

TensorBoard 是 TensorFlow 的可视化工具，可以：
- 监控标量指标（损失、准确率）
- 可视化模型架构
- 查看梯度/权重直方图
- 嵌入向量投影

In [ ]:
# TensorBoard 回调的使用
import os
log_dir = os.path.join(os.getcwd(), "logs", "mnist_training")

tensorboard = keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,  # 每 1 轮记录一次权重直方图
)

model = get_mnist_model()
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    x_train, y_train,
    epochs=3,
    validation_data=(x_val, y_val),
    callbacks=[tensorboard]
)

print(f"\nTensorBoard 日志目录: {log_dir}")
print("\n启动 TensorBoard 的方式：")
print("  终端命令:  tensorboard --logdir " + log_dir)
print("  Jupyter:   %load_ext tensorboard")
print("             %tensorboard --logdir " + log_dir)

### 第二阶段小结

| 步骤 | API | 作用 |
|------|-----|------|
| 编译 | `compile(optimizer, loss, metrics)` | 配置训练规则 |
| 训练 | `fit(x, y, epochs, batch_size, validation_data)` | 训练模型 |
| 评估 | `evaluate(x, y)` | 评估性能 |
| 预测 | `predict(x)` | 预测新数据 |
| 扩展 | 自定义 Metric / Callback | 扩展 fit() 能力 |

---
# 总结

## Keras 核心工作流

```
1. 定义层 / 搭建模型
   - Sequential：简单的线性堆叠
   - 函数式 API：复杂结构（残差、多输入输出）← 推荐
   - 模型子类化：完全控制 call() 方法

2. compile(optimizer, loss, metrics)
   - 配置优化器、损失函数、监控指标

3. fit(x, y, epochs, batch_size, validation_data, callbacks)
   - 训练模型

4. evaluate() / predict()
   - 评估和预测

5. 扩展 fit() 能力
   - 自定义 Metric → 继承 keras.metrics.Metric
   - 自定义 Callback → 继承 keras.callbacks.Callback
   - 完全自定义训练 → 打开 7.4-自定义训练循环.ipynb
```

---
**下一步学习**：打开 `7.4-自定义训练循环.ipynb` 学习如何用 GradientTape 和 tf.function 编写完全自定义的训练流程。